# 06 — Cross-Notebook Model Evaluation

Consolidate and compare forecasting results across all four data layers:
- **02**: Budget Functions (20 functions, FY2017–2024, cumulative YTD)
- **03**: Agencies (116 agencies, FY2017–2024, cumulative YTD)
- **04**: Federal Accounts (2,236 accounts, FY2017–2024, cumulative YTD)
- **05**: Geography (56 states/territories, FY2008–2024, quarterly spending)

**Central finding:** Data structure (cumulative YTD vs quarterly) determines model ranking more than data size or series count.

## 1. Setup & Load All Forecast Files

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.spines.top': False, 'axes.spines.right': False})

ROOT         = Path('..').resolve()
FORECAST_DIR = ROOT / 'pipeline-a-hierarchical' / 'data' / 'forecasts'

# Load all per-notebook metrics
bf_metrics  = pd.read_csv(FORECAST_DIR / 'budget_functions_model_metrics.csv')
ag_metrics  = pd.read_csv(FORECAST_DIR / 'agency_model_metrics.csv')
fa_metrics  = pd.read_csv(FORECAST_DIR / 'federal_accounts_model_metrics.csv')
geo_metrics = pd.read_csv(FORECAST_DIR / 'geography_model_metrics.csv')

# Load total predictions for residual plots
bf_preds  = pd.read_csv(FORECAST_DIR / 'budget_functions_total_predictions.csv', parse_dates=['ds'])
ag_preds  = pd.read_csv(FORECAST_DIR / 'agency_total_predictions.csv',           parse_dates=['ds'])
fa_preds  = pd.read_csv(FORECAST_DIR / 'federal_accounts_total_predictions.csv', parse_dates=['ds'])
geo_preds = pd.read_csv(FORECAST_DIR / 'geography_total_predictions.csv',        parse_dates=['ds'])

print('Forecast files loaded.')
for name, df in [('Budget Functions', bf_metrics), ('Agency', ag_metrics),
                 ('Federal Accounts', fa_metrics), ('Geography', geo_metrics)]:
    print(f'\n{name}:')
    print(df.to_string(index=False))

All four metrics files are loaded. Each contains the MAE, RMSE, and MAPE for Prophet, SARIMA, and XGBoost on the total-series forecast for that data layer. These are the numbers we'll compare across notebooks to identify consistent patterns and data-structure effects.

## 2. Total-Series MAPE Comparison — All 4 Notebooks

In [ ]:
# Build consolidated comparison table
layers = [
    ('Budget Functions', bf_metrics,  'Cumulative YTD', 'FY2023–2024',  8),
    ('Agency',           ag_metrics,  'Cumulative YTD', 'FY2023–2024',  8),
    ('Federal Accounts', fa_metrics,  'Cumulative YTD', 'FY2023–2024',  8),
    ('Geography',        geo_metrics, 'Quarterly',      'FY2021–2024', 16),
]

rows = []
for layer, met, dtype, test_period, n_test in layers:
    for _, r in met.iterrows():
        rows.append({
            'Layer':       layer,
            'Data Type':   dtype,
            'Test Period': test_period,
            'Test Qtrs':   n_test,
            'Model':       r['Model'],
            'MAE ($B)':    r['MAE ($B)'],
            'RMSE ($B)':   r['RMSE ($B)'],
            'MAPE (%)':    r['MAPE (%)']
        })

summary = pd.DataFrame(rows)
print('=== CONSOLIDATED MODEL COMPARISON — TOTAL SERIES MAPE ===')
pivot = summary.pivot_table(index=['Layer','Data Type'], columns='Model', values='MAPE (%)')
pivot['Best Model'] = pivot.idxmin(axis=1)
print(pivot.round(1).to_string())

The pivot table makes the data-structure effect immediately visible: SARIMA dominates on all three cumulative YTD layers at 3.8% MAPE each, while Prophet dominates on geography (13.7% MAPE). The pattern holds perfectly — every cumulative YTD layer has the same SARIMA winner and every quarterly layer has a Prophet winner. This is not a coincidence; it reflects a structural property of how each model handles within-year accumulation.

In [ ]:
# Grouped bar chart: MAPE by model and data layer
models  = ['Prophet', 'SARIMA', 'XGBoost']
colors  = {'Prophet': 'orange', 'SARIMA': 'green', 'XGBoost': 'purple'}
layer_labels = ['Budget\nFunctions', 'Agency', 'Federal\nAccounts', 'Geography']

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(4)
w = 0.25
for i, model in enumerate(models):
    mapes = [summary[(summary['Layer']==l) & (summary['Model']==model)]['MAPE (%)'].values[0]
             for l in ['Budget Functions','Agency','Federal Accounts','Geography']]
    bars = ax.bar(x + i*w, mapes, w, label=model, color=colors[model], alpha=0.85)
    for bar, v in zip(bars, mapes):
        if v < 100:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    f'{v:.1f}%', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x + w)
ax.set_xticklabels(layer_labels, fontsize=10)
ax.set_ylabel('MAPE (%) — lower is better')
ax.set_title('Model Comparison Across All Data Layers (Total Series MAPE)')
ax.legend(fontsize=9)
ax.set_ylim(0, 70)
# Annotate data type
for xi, label in zip(x, ['YTD','YTD','YTD','Quarterly']):
    ax.text(xi + w, -5, label, ha='center', fontsize=8, color='gray', style='italic')
plt.tight_layout()
plt.show()

The bar chart shows the model ranking clearly across all four data layers. SARIMA's green bars are the shortest (best) for the three YTD layers and the tallest (worst) for Geography. Prophet's orange bars show the inverse pattern. XGBoost (purple) is consistently in the middle — never the best, never the worst — confirming its role as a reliable fallback across all data structures.

## 3. Why Data Structure Determines Model Ranking

In [ ]:
# Illustrate the YTD vs quarterly difference using actual data
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: YTD cumulative pattern (Budget Functions — FY2022 and FY2023)
ax = axes[0]
for fy in [2021, 2022, 2023]:
    sub = bf_preds[bf_preds['fy'] == fy] if fy in bf_preds['fy'].values else None
    # Load from full dataset
from pathlib import Path
import pandas as pd
CLEAN_A = ROOT / 'pipeline-a-hierarchical' / 'data' / 'cleaned'
bf_raw = pd.read_csv(CLEAN_A / 'budget_functions_quarterly_all.csv', dtype={'budget_function_id': str})
bf_raw = bf_raw.dropna(subset=['budget_function_id'])
bf_total = bf_raw.groupby(['fy','quarter'])['obligated_amount'].sum().reset_index()

for fy, color in [(2021,'steelblue'),(2022,'orange'),(2023,'green')]:
    sub = bf_total[bf_total['fy']==fy]
    axes[0].plot(sub['quarter'], sub['obligated_amount']/1e12, 'o-', ms=5, label=f'FY{fy}', color=color)
axes[0].set_title('Cumulative YTD — Budget Functions\n(Q4 always > Q3 > Q2 > Q1)')
axes[0].set_xlabel('Quarter')
axes[0].set_ylabel('Obligated Amount ($T)')
axes[0].set_xticks([1,2,3,4])
axes[0].legend(fontsize=8)

# Right: Quarterly spending pattern (Geography — same FYs)
GEO_CLEAN = ROOT / 'pipeline-b-geography' / 'data' / 'basic-geography' / 'cleaned'
geo_raw = pd.read_csv(GEO_CLEAN / 'geography_state_all_FY2008_2024.csv')
geo_total = geo_raw.groupby(['fy','quarter'])['obligated_amount'].sum().reset_index()
for fy, color in [(2021,'steelblue'),(2022,'orange'),(2023,'green')]:
    sub = geo_total[geo_total['fy']==fy]
    axes[1].plot(sub['quarter'], sub['obligated_amount']/1e12, 'o-', ms=5, label=f'FY{fy}', color=color)
axes[1].set_title('Quarterly Spending — Geography\n(No accumulation — Q1 ≈ Q2 ≈ Q3 ≈ Q4)')
axes[1].set_xlabel('Quarter')
axes[1].set_ylabel('Obligated Amount ($T)')
axes[1].set_xticks([1,2,3,4])
axes[1].legend(fontsize=8)

plt.suptitle('Cumulative YTD vs Quarterly Spending — The Key Structural Difference', fontsize=11)
plt.tight_layout()
plt.show()

**Left chart**: Cumulative YTD spending always rises within each fiscal year — Q4 is always the highest, Q1 the lowest. SARIMA's seasonal differencing (removing Q1 FY2023 minus Q1 FY2022, Q2 minus Q2, etc.) cleanly isolates the year-over-year growth signal, which is what we want to model.

**Right chart**: Geography quarterly spending fluctuates within the year — Q1 and Q4 are roughly the same scale, with no monotonic accumulation. SARIMA's differencing removes genuine signal rather than accumulation noise, destroying the model's ability to track the level. Prophet, which fits a trend plus a seasonal pattern independently, handles both structures but is better calibrated for the quarterly case.

## 4. Residual Analysis — Error Patterns Across Notebooks

In [ ]:
# Residual plots for best model per layer
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

configs = [
    (bf_preds,  'Budget Functions', 'sarima',  'SARIMA (3.8%)',  'green',  axes[0,0]),
    (ag_preds,  'Agency',           'sarima',  'SARIMA (3.8%)',  'green',  axes[0,1]),
    (fa_preds,  'Federal Accounts', 'sarima',  'SARIMA (3.8%)',  'green',  axes[1,0]),
    (geo_preds, 'Geography',        'prophet', 'Prophet (13.7%)', 'orange', axes[1,1]),
]

for preds, title, model_col, model_label, color, ax in configs:
    residuals = (preds[model_col] - preds['actual']) / 1e9
    quarters  = [f"FY{int(r.fy)}Q{int(r.quarter)}" for _, r in preds.iterrows()]
    colors_bar = ['red' if v < 0 else color for v in residuals]
    ax.bar(range(len(residuals)), residuals, color=colors_bar, alpha=0.8)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_xticks(range(len(quarters)))
    ax.set_xticklabels(quarters, rotation=45, ha='right', fontsize=7)
    ax.set_ylabel('Error ($B)  [pred − actual]')
    ax.set_title(f'{title} — {model_label}')
    ax.axhline(residuals.mean(), color='navy', lw=1, ls='--',
               label=f'Mean error: ${residuals.mean():.0f}B')
    ax.legend(fontsize=7)

plt.suptitle('Residuals — Best Model per Data Layer (pred − actual, $B)', fontsize=11)
plt.tight_layout()
plt.show()

Residual plots reveal error bias. A good model has residuals balanced around zero (alternating red and colored bars) with no systematic drift. SARIMA on the three YTD layers shows exactly this — errors flip between positive and negative with no trend, confirming the model captured the spending level correctly. The Geography / Prophet residuals show more variation (larger bars) but also no systematic directional drift, indicating the model tracks the level without a persistent over- or under-estimate bias.

## 5. XGBoost Consistency Analysis

In [ ]:
# XGBoost MAPE across all layers
xgb_rows = summary[summary['Model'] == 'XGBoost'][['Layer','Data Type','MAPE (%)','MAE ($B)','RMSE ($B)']]
print('=== XGBoost Performance Across All Layers ===')
print(xgb_rows.to_string(index=False))

# Feature importance summary from each notebook (hardcoded from results)
feat_imp = pd.DataFrame([
    {'Notebook': 'Budget Functions', 'lag_4': 43.2, 'lag_8': 27.3, 'lag_1':  9.1, 'roll4_mean':  7.4, 'quarter': 6.1, 'covid': 1.1},
    {'Notebook': 'Agency',           'lag_4': 41.3, 'lag_8': 35.4, 'lag_1':  7.1, 'roll4_mean':  6.9, 'quarter': 3.9, 'covid': 0.6},
    {'Notebook': 'Federal Accounts', 'lag_4':  4.1, 'lag_8': 16.4, 'lag_1': 12.2, 'roll4_mean': 44.4, 'quarter':10.1, 'covid': 0.5},
    {'Notebook': 'Geography',        'lag_4': 16.4, 'lag_8':  2.7, 'lag_1': 16.3, 'roll4_mean': 49.2, 'quarter': 1.7, 'covid': 2.6},
])
print('\n=== XGBoost Feature Importance (%) Across Notebooks ===')
print(feat_imp.set_index('Notebook').round(1).to_string())

XGBoost delivered consistent mid-range performance: 8.3% (budget functions), 5.4% (agency), 5.8% (federal accounts), 21.5% (geography). It was never the best model but never catastrophically wrong — a reliable fallback across all data structures.

Feature importance shifted systematically with data structure:
- **YTD cumulative data**: `lag_4` dominates (41–43%) — the same quarter last year is the strongest predictor because YTD values are highly regular year-over-year
- **Quarterly data (geography)** and **heterogeneous accounts**: `roll4_mean` dominates (44–49%) — the smoothed recent average is more robust when individual quarters are noisy or data is sparse

## 6. Per-Series Winner Summary

In [ ]:
# Load per-series predictions and compute winner per series
from sklearn.metrics import mean_absolute_error

results_per_series = []

# Budget functions per-function
bf_series = pd.read_csv(FORECAST_DIR / 'budget_functions_per_function_predictions.csv', parse_dates=['ds'])
for fid, grp in bf_series.groupby('budget_function_id'):
    p = grp[grp['model']=='Prophet']; s = grp[grp['model']=='SARIMA']
    if len(p) and len(s):
        mp = np.mean(np.abs((p['predicted'].values-p['actual'].values)/np.where(p['actual'].values!=0,p['actual'].values,np.nan)))*100
        ms = np.mean(np.abs((s['predicted'].values-s['actual'].values)/np.where(s['actual'].values!=0,s['actual'].values,np.nan)))*100
        results_per_series.append({'Layer':'Budget Functions','Series':grp['function_name'].iloc[0][:30],'Prophet MAPE':round(mp,1),'SARIMA MAPE':round(ms,1),'Winner':'Prophet' if mp<ms else 'SARIMA'})

# Agency per-agency
ag_series = pd.read_csv(FORECAST_DIR / 'agency_per_agency_predictions.csv', parse_dates=['ds'])
for aid, grp in ag_series.groupby('agency_id'):
    p = grp[grp['model']=='Prophet']; s = grp[grp['model']=='SARIMA']
    if len(p) and len(s):
        actual_p = p['actual'].values; actual_s = s['actual'].values
        mask_p = actual_p != 0; mask_s = actual_s != 0
        if mask_p.sum()>0 and mask_s.sum()>0:
            mp = np.mean(np.abs((p['predicted'].values[mask_p]-actual_p[mask_p])/actual_p[mask_p]))*100
            ms = np.mean(np.abs((s['predicted'].values[mask_s]-actual_s[mask_s])/actual_s[mask_s]))*100
            if mp < 1e6 and ms < 1e6:
                results_per_series.append({'Layer':'Agency','Series':grp['agency_name'].iloc[0][:30],'Prophet MAPE':round(mp,1),'SARIMA MAPE':round(ms,1),'Winner':'Prophet' if mp<ms else 'SARIMA'})

# Geography per-state
geo_series = pd.read_csv(FORECAST_DIR / 'geography_per_state_predictions.csv', parse_dates=['ds'])
for code, grp in geo_series.groupby('geo_code'):
    p = grp[grp['model']=='Prophet']; s = grp[grp['model']=='SARIMA']
    if len(p) and len(s):
        actual_p = p['actual'].values; actual_s = s['actual'].values
        mask_p = actual_p != 0; mask_s = actual_s != 0
        if mask_p.sum()>0 and mask_s.sum()>0:
            mp = np.mean(np.abs((p['predicted'].values[mask_p]-actual_p[mask_p])/actual_p[mask_p]))*100
            ms = np.mean(np.abs((s['predicted'].values[mask_s]-actual_s[mask_s])/actual_s[mask_s]))*100
            results_per_series.append({'Layer':'Geography','Series':grp['state_name'].iloc[0],'Prophet MAPE':round(mp,1),'SARIMA MAPE':round(ms,1),'Winner':'Prophet' if mp<ms else 'SARIMA'})

per_series_df = pd.DataFrame(results_per_series)
winner_counts = per_series_df.groupby(['Layer','Winner']).size().unstack(fill_value=0)
print('=== Per-Series Model Winner Count ===')
print(winner_counts.to_string())
print(f'\nOverall: Prophet wins {(per_series_df["Winner"]=="Prophet").sum()}/{len(per_series_df)} series ({(per_series_df["Winner"]=="Prophet").mean()*100:.0f}%)')

The per-series winner count shows how consistently each model wins at the individual series level (excluding diverged SARIMA results). Prophet's advantage is most pronounced in Geography (8/10 states) and Budget Functions, while SARIMA shows strength only in a few individual series. For the dashboard, this means Prophet should be the default per-series model with SARIMA reserved for the specific series where it was measured to outperform.

## 7. Final Model Recommendation Matrix

In [ ]:
# Recommendation matrix
rec = pd.DataFrame([
    {'Data Layer':     'Budget Functions',
     'Total Series':  'SARIMA (3.8%)',
     'Per Series':    'Prophet (wins 7/10)',
     'Fallback':      'XGBoost (8.3%)',
     'Flagged':       'Income Security, Commerce & Housing Credit'},
    {'Data Layer':     'Agency',
     'Total Series':  'SARIMA (3.8%)',
     'Per Series':    'Prophet (wins 10/10)',
     'Fallback':      'XGBoost (5.4%)',
     'Flagged':       'Labor, Transportation, Education (both models poor)'},
    {'Data Layer':     'Federal Accounts',
     'Total Series':  'SARIMA (3.8%)',
     'Per Series':    'Prophet (wins 14/17); SARIMA for Hosp. Ins., Mil. Retirement, Tax Refunds',
     'Fallback':      'XGBoost (5.8%)',
     'Flagged':       'COVID Payments, PPP Loans, Student Loans, Unemployment (both models fail)'},
    {'Data Layer':     'Geography (States)',
     'Total Series':  'Prophet (13.7%)',
     'Per Series':    'Prophet (wins 8/10); SARIMA for TX, VA only',
     'Fallback':      'XGBoost (21.5%)',
     'Flagged':       'Florida (both models poor > 50% MAPE)'},
])

print('=== DASHBOARD MODEL ASSIGNMENT ===')
for _, row in rec.iterrows():
    print(f"\n{row['Data Layer']}:")
    print(f"  Total:    {row['Total Series']}")
    print(f"  Per-item: {row['Per Series']}")
    print(f"  Fallback: {row['Fallback']}")
    print(f"  Flag:     {row['Flagged']}")

**The final decision rule for the dashboard:**

1. **Use SARIMA** for all total-series views on hierarchical data (budget functions, agencies, federal accounts) — 3.8% MAPE consistently
2. **Use Prophet** for all per-series views (per-function, per-agency, per-account, per-state) as the default — it wins the majority of individual comparisons and never catastrophically diverges
3. **Override to SARIMA** for the 3 specific federal accounts and 2 states where SARIMA was measured to beat Prophet
4. **Use XGBoost** for the Geography total series as the second option (21.5%), and as the general fallback when neither Prophet nor SARIMA can be trusted
5. **Flag as high-uncertainty** any series with MAPE > 100% from both models — display actual data but suppress the forecast line or show it with a warning label

## 8. Key Insights Summary

In [ ]:
# Summary visualization: MAPE heatmap across models and layers
mape_matrix = summary.pivot_table(index='Model', columns='Layer', values='MAPE (%)')
# Order columns and rows
mape_matrix = mape_matrix[['Budget Functions','Agency','Federal Accounts','Geography']]
mape_matrix = mape_matrix.loc[['SARIMA','Prophet','XGBoost']]

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(mape_matrix.values, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=60)
plt.colorbar(im, ax=ax, label='MAPE (%)')
ax.set_xticks(range(4))
ax.set_xticklabels(['Budget\nFunctions','Agency','Federal\nAccounts','Geography'], fontsize=10)
ax.set_yticks(range(3))
ax.set_yticklabels(['SARIMA','Prophet','XGBoost'], fontsize=11)
for i in range(3):
    for j in range(4):
        v = mape_matrix.values[i, j]
        ax.text(j, i, f'{v:.1f}%', ha='center', va='center', fontsize=11,
                color='white' if v > 30 else 'black', fontweight='bold')
ax.set_title('MAPE Heatmap — All Models × All Data Layers\n(green = best, red = worst)', fontsize=11)
plt.tight_layout()
plt.show()

print('\n=== PROJECT-WIDE KEY INSIGHTS ===')
insights = [
    '1. DATA STRUCTURE > DATA SIZE: Cumulative YTD → SARIMA wins. Quarterly → Prophet wins.',
    '2. SARIMA: 3.8% MAPE on all 3 YTD total series (identical). Fails on geography (58.3%).',
    '3. PROPHET: 13.7% on geography total. Wins per-series across all 4 notebooks.',
    '4. XGBOOST: Consistent 5–22% across all notebooks — best fallback, needs no per-series fitting.',
    '5. COVID IMPACT: Series touched by pandemic stimulus (SBA PPP, UI, Student Loans) are unforecastable from history alone.',
    '6. SSA Old-Age Trust Fund: 2.8% Prophet MAPE — most predictable series in the project.',
    '7. Florida: Both models fail (>50% MAPE) — hurricane + Medicaid shocks make it structurally unpredictable.',
    '8. XGBoost feature shift: lag_4 dominates on YTD data; roll4_mean dominates on quarterly/sparse data.',
]
for insight in insights:
    print(f'  {insight}')

The MAPE heatmap is the single most informative chart in the project. Green cells mark where a model excels; red cells mark where it fails. The pattern is stark: SARIMA has a green column for YTD data and a red cell for geography. Prophet is the inverse. XGBoost is yellow-green throughout — consistently decent, never outstanding.

**8 key insights from this project:**

1. **Data structure determines the winner** — not model complexity, not data volume
2. **SARIMA achieves identical 3.8% MAPE** on budget functions, agencies, and federal accounts because they are all the same aggregate YTD series viewed from different angles
3. **Prophet wins geography** because quarterly spending has no within-year accumulation for SARIMA to exploit
4. **XGBoost is the universal fallback** — 5–22% across all notebooks with zero per-series fitting
5. **COVID-era programs are structurally unforecastable** from historical data — these require a policy-driven estimate, not a statistical model
6. **The SSA Old-Age Trust Fund** (Prophet 2.8% MAPE) is the most predictable federal account — demographic-driven, formula-bound, and insulated from discretionary policy changes
7. **Florida is the most anomalous state** — hurricane disaster relief creates irregular spending shocks that no historical model can anticipate
8. **XGBoost feature importance** shifts from lag_4-dominant (YTD, regular series) to roll4_mean-dominant (quarterly, heterogeneous series) — the model automatically adapts its signal weighting to the data structure